In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [4]:

#model = "claude-haiku-4-5"
#model = "claude-sonnet-4-6"
#model = "claude-haiku-4-5-20251001"

In [2]:
def add_user_messages(messages, text):
    user_message  = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_messages(messages, text):
    assistant_message  = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [24]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "json or python or regex"
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_messages(messages, prompt)
    add_assistant_messages(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [25]:
dataset = generate_dataset()

with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [27]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:
{test_case["task"]}
* Respond only with Python, JSON, or plain Regex
* Do not add any comments or commentary or explanation
"""
    messages = []
    add_user_messages(messages, prompt)
    add_assistant_messages(messages, "```code")
    output = chat(messages, stop_sequences=["```"])
    return output

In [15]:
# Function to grade a test case + output using a model
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    messages = []
    add_user_messages(messages, eval_prompt)
    add_assistant_messages(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [23]:
# Functions to validate the output structure
import re
import ast


def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0


def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0


def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0


def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)

In [16]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    # TO DO
    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    syntax_score = grade_syntax(output, test_case)

    score = (model_score + syntax_score) / 2

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning
    }

In [21]:
from statistics import mean

def run_eval(dataset):
    """Loads the datset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average Score: {average_score}")
    
    return results

In [22]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

Average Score: 7


In [19]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS S3 Bucket Name Validation Regex\n\nHere's a comprehensive solution with multiple approaches:\n\n## Solution 1: Single Regex Pattern (Recommended)\n\n```regex\n^(?!.*\\.\\.)[a-z0-9]([a-z0-9.-]*[a-z0-9])?$\n```\n\n**Applied with length constraint (3-63 characters):**\n\n```regex\n^(?!.*\\.\\.)[a-z0-9]([a-z0-9.-]{1,61}[a-z0-9])?$\n```\n\n## Solution 2: Complete Implementation Examples\n\n### JavaScript\n```javascript\nfunction validateS3BucketName(bucketName) {\n  const pattern = /^(?!.*\\.\\.)([a-z0-9]([a-z0-9.-]{0,61}[a-z0-9])?|[a-z0-9])$/;\n  \n  // Check length constraint\n  if (bucketName.length < 3 || bucketName.length > 63) {\n    return false;\n  }\n  \n  return pattern.test(bucketName);\n}\n\n// Test cases\nconsole.log(validateS3BucketName(\"my-bucket\"));           // true\nconsole.log(validateS3BucketName(\"my.bucket\"));           // true\nconsole.log(validateS3BucketName(\"my--bucket\"));          // true\nconsole.log(validateS3BucketName(\"-my-buck